# Medical Assistant - Part 2: Fine-tuning

Fine-tune Phi-3 Mini on medical data using LoRA + Unsloth.

## 1 - Setup

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install transformers datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-d9uhezv3/unsloth_249eb2d814db420dad143b0f6c8909a7
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-d9uhezv3/unsloth_249eb2d814db420dad143b0f6c8909a7
  Resolved https://github.com/unslothai/unsloth.git to commit b3640802253f64117ee228718be7fab32e47aa5f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 117.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.4 MB/s eta 0:00:0

In [2]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset
from google.colab import drive
from transformers import TrainingArguments, TextStreamer
from trl import SFTTrainer
import json
import os
import torch

drive.mount('/content/drive')

# Set working directory with defensive programming
work_dir = '/content/drive/MyDrive/Colab Notebooks/ana-barbosa-modulo-03'
if not os.path.exists(work_dir):
    print(f"⚠️  Creating missing directory: {work_dir}")
    os.makedirs(work_dir, exist_ok=True)
    print(f"✅ Directory created")

os.chdir(work_dir)
print(f"✅ Working directory: {os.getcwd()}")

# Check GPU
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"CUDA available: {torch.cuda.is_available()}")

print("✅ Setup complete")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Mounted at /content/drive
✅ Working directory: /content/drive/MyDrive/Colab Notebooks/ana-barbosa-modulo-03
GPU: NVIDIA L4
CUDA available: True
✅ Setup complete


## 2 - Load Prepared Data

In [3]:
# Load training data
with open("data/processed/medical_data_train.jsonl", "r") as f:
    train_data = [json.loads(line) for line in f]

# Load validation data
with open("data/processed/medical_data_val.jsonl", "r") as f:
    val_data = [json.loads(line) for line in f]

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"\nExample: {train_data[0]['text'][:200]}...")

Training samples: 20712
Validation samples: 5178

Example: Diagnosis: Acute viral pharyngitis (disorder)...


## 3 - Load Model and Tokenizer

In [4]:
model_name = "microsoft/Phi-3-mini-4k-instruct"
max_seq_length = 2048

print(f"Loading {model_name}...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=torch.float16 if is_bfloat16_supported() == False else torch.bfloat16,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,                                      # Rank
    lora_alpha=16,                            # Scaling
    lora_dropout=0.05,                        # Dropout
    target_modules=["qkv_proj", "o_proj"],    # Phi-3 attention layers
    bias="none",
    use_gradient_checkpointing="unsloth",     # Faster gradient checkpointing
    random_state=42,
)

# Check trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
percent = 100 * trainable / total

print(f"✅ Model loaded with LoRA")
print(f"Trainable parameters: {trainable/1e6:.1f}M ({percent:.2f}%)")
print(f"Total parameters: {total/1e9:.2f}B")

Loading microsoft/Phi-3-mini-4k-instruct...
==((====))==  Unsloth 2026.5.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!


Unsloth 2026.5.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


✅ Model loaded with LoRA
Trainable parameters: 1.6M (0.08%)
Total parameters: 2.01B


## 4 - Prepare Data

In [5]:
# Load training and validation data
with open("data/processed/medical_data_train.jsonl", "r") as f:
    train_data = [json.loads(line) for line in f]

with open("data/processed/medical_data_val.jsonl", "r") as f:
    val_data = [json.loads(line) for line in f]

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

EOS_TOKEN = tokenizer.eos_token

# Add EOS token to each sample
def add_eos(examples):
    return {"text": [text + EOS_TOKEN for text in examples["text"]]}

train_dataset = Dataset.from_dict({"text": [item["text"] for item in train_data]})
val_dataset = Dataset.from_dict({"text": [item["text"] for item in val_data]})

train_dataset = train_dataset.map(add_eos, batched=True)
val_dataset = val_dataset.map(add_eos, batched=True)

print(f"✅ Datasets ready")
print(f"Example: {train_dataset[0]['text'][:300]}...")

Training samples: 20712
Validation samples: 5178


Map:   0%|          | 0/20712 [00:00<?, ? examples/s]

Map:   0%|          | 0/5178 [00:00<?, ? examples/s]

✅ Datasets ready
Example: Diagnosis: Acute viral pharyngitis (disorder)<|endoftext|>...


## 5 - Configure Training Arguments

In [6]:
# Detect dynamic dtype
dtype = torch.float16 if is_bfloat16_supported() == False else torch.bfloat16

# Training arguments optimized for T4 GPU
training_args = TrainingArguments(
    output_dir="outputs",
    max_steps=500,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=1,
    optim="adamw_8bit",
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

print("✅ Training arguments configured")

✅ Training arguments configured


## 6 - Create Trainer

In [7]:
# Create SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    dataset_num_proc=2,
    packing=False,
    max_seq_length=max_seq_length,
    dataset_text_field="text",
)

print("✅ Trainer created")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/20712 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/5178 [00:00<?, ? examples/s]

✅ Trainer created


## 7 - Train!

In [8]:
print("Starting training...")

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Final loss: {train_result.training_loss:.4f}")

Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,712 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 1,572,864 of 3,822,652,416 (0.04% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
500,1.425809,1.165606


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Training complete!
Final loss: 1.4968


## 8 - Save Model

In [10]:
# Save LoRA adapters
model.save_pretrained("models/medical_assistant_lora")
tokenizer.save_pretrained("models/medical_assistant_lora")

print("✅ Model saved to models/medical_assistant_lora")

Unsloth: Restored added_tokens_decoder metadata in models/medical_assistant_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in models/medical_assistant_lora.


✅ Model saved to models/medical_assistant_lora


## 9 - Test the Model

In [17]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test prompts
test_prompts = [
    "What is hypertension?",
    "How do you treat diabetes?",
    "What are the symptoms of pneumonia?",
]

for prompt in test_prompts:
    formatted = f"Question: {prompt}\nAnswer:"
    print(f"\nPrompt: {prompt}\n")

    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    text_streamer = TextStreamer(tokenizer, skip_special_tokens=True)

    with torch.no_grad():
        model.generate(
            **inputs,
            streamer=text_streamer,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
        )

    print("\n" + "-" * 70)


Prompt: What is hypertension?

Question: What is 

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


hypertension?
Answer: Hypertension is a medical condition in which the blood pressure in the arteries is persistently elevated. It is also known as high blood pressure. Hypertension is a major risk factor for cardiovascular disease, stroke, and kidney failure. It is the most common cardiovascular disease in the world. Hypertension is a major risk factor for cardiovascular disease, stroke, and kidney failure. It is the most common cardiovascular disease in the world. Hypertension is a major risk factor for cardiovascular disease, stroke, and kidney failure. It is the most common cardiovascular disease in the world. Hypertension is a major risk factor for cardiovascular disease, stroke, and kidney failure. It is the most common cardiovascular disease in the world. Hypertension is a major risk factor for cardiovascular disease, stroke, and kidney failure. It is the most common cardiovascular disease in the world. Hypertension is a major risk factor for cardiovascular disease, stroke, and 

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


diabetes?
Answer: Treatment of diabetes depends on the type of diabetes, the severity of the disease, and the patient's overall health. Treatment may include:
- Lifestyle changes: A healthy diet and regular exercise can help control diabetes.
- Medications: Oral medications are used to treat type 2 diabetes. Insulin is used to treat type 1 diabetes and sometimes type 2 diabetes.
- Insulin therapy: Insulin is used to treat type 1 diabetes and sometimes type 2 diabetes. Insulin is usually given by injection.
- Monitoring: Blood glucose levels are monitored to make sure that the treatment is working.
- Education: Patients and their families are educated about diabetes and how to manage it.

----------------------------------------------------------------------

Prompt: What are the symptoms of pneumonia?

Question: What are the symptoms of 

Both `max_new_tokens` (=300) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


pneumonia?
Answer: Pneumonia symptoms may include:
- Chest pain
- Cough
- Fever
- Headache
- Loss of appetite
- Muscle aches
- Shortness of breath

----------------------------------------------------------------------


## Done!

✅ Fine-tuning complete!

Model saved to: `models/medical_assistant_lora/`